In [1]:
import pandas as pd
import numpy as np

%store -r df
df['h-cleaned'] = df['h-cleaned'].diff()
df.head()

,TICKER,DATETIME,CLOSE,h-cleaned
0,SiZ6SiH7 [FUTSPREAD],2026-10-01 07:00:00,1343.0,NaN
1,SiZ6SiH7 [FUTSPREAD],2026-10-01 07:15:00,1334.0,-9.0
2,SiZ6SiH7 [FUTSPREAD],2026-10-01 07:30:00,1330.0,-4.0
3,SiZ6SiH7 [FUTSPREAD],2026-10-01 07:45:00,1330.0,0.0
4,SiZ6SiH7 [FUTSPREAD],2026-10-01 08:00:00,1325.0,-5.0
...,...,...,...,...
1511,SiZ6SiH7 [FUTSPREAD],2026-10-28 22:45:00,1443.0,0.0
1512,SiZ6SiH7 [FUTSPREAD],2026-10-28 23:00:00,1444.0,1.0
1513,SiZ6SiH7 [FUTSPREAD],2026-10-28 23:15:00,1440.0,-4.0
1514,SiZ6SiH7 [FUTSPREAD],2026-10-28 23:30:00,1436.0,-4.0


In [3]:
from statsmodels.tsa.stattools import adfuller, kpss

adf_result = adfuller(df['h-cleaned'].dropna(), regression='c', result_object=True)
kpss_result = kpss(df['h-cleaned'].dropna(), regression='c', result_object=True )
print('--------- ADF - test  -------------- ')
print(adf_result)
print('--------- KPSS - test  -------------- ')
print(kpss_result)

--------- ADF - test  -------------- 
ADFullerResult
ADF Statistic: -16.05083
P-value: 0.00000
Used Lag: 7
Nobs: 1507
Critical Values: {'1%': np.float64(-3.434696697851678), '5%': np.float64(-2.8634597924260303), '10%': np.float64(-2.567792072971565)}

--------- KPSS - test  -------------- 
KPSSResult
KPSS Statistic: 0.20227
P-value: 0.10000
Lags: 19
Critical Values: {'10%': 0.347, '5%': 0.463, '2.5%': 0.574, '1%': 0.739}



C:\Users\user\AppData\Local\Temp\ipykernel_3036\1112846388.py:4: InterpolationWarning: The test statistic is outside of the range of p-values available in the look-up table. The actual p-value is greater than the p-value returned.
  kpss_result = kpss(df['h-cleaned'].dropna(), regression='c', result_object=True )


In [4]:
# функции для работы с сырыми массивами (raw=True)
def get_adf_p(x):
    if np.all(x == x[0]):
        return np.nan
    try:
        return adfuller(x, regression="c", autolag="AIC", result_object=True).pvalue
    except:
        return np.nan

def get_kpss_p(x):
    if np.all(x == x[0]):
        return np.nan
    try:
        return kpss(x, regression="c", nlags="auto", result_object=True).pvalue
    except:
        return np.nan

def stationarity_rolling_window(df, column_name, min_pct=0.20, max_pct=0.70, step_pct=0.05, alpha_adf=0.05, alpha_kpss=0.05):
    series = df[column_name].dropna()
    total_len = len(series)

    min_window = int(total_len * min_pct)
    max_window = int(total_len * max_pct)
    step = max(1, int(total_len * step_pct))
    window_sizes = range(min_window, max_window + 1, step)

    print(f"Анализ ряда длиной {total_len}")

    successful_windows = [] # для сводной таблицы по p-value тестов

    for window_size in window_sizes:
        adf_series = series.rolling(window=window_size).apply(get_adf_p, raw=True)
        kpss_series = series.rolling(window=window_size).apply(get_kpss_p, raw=True)

        # иаска для выполнений условий стационарности, применяемая к рядам
        valid_mask = (adf_series < alpha_adf) & (kpss_series >= alpha_kpss)

        # Сохраняем столбец ADF в итоговый датафрейм (только валидные значения, остальные NaN)
        df[f'ADF {window_size}'] = adf_series.where(valid_mask)
        df[f'KPSS {window_size}'] = kpss_series.where(valid_mask)

        # информация для таблицы results_df только по успешным точкам
        end_indices = series.index[valid_mask]
        for end_label in end_indices:
            # найти порядковый номер конечной точки в исходном ряду
            end_idx = series.index.get_loc(end_label)
            start_idx = end_idx - window_size + 1
            start_label = series.index[start_idx]

            successful_windows.append({
                "window_size": window_size,
                "window_pct": round(window_size / total_len, 2),
                "start_index": start_label,
                "end_index": end_label,
                "start_datetime": df.loc[start_label, "DATETIME"] if "DATETIME" in df.columns else start_label,
                "end_datetime": df.loc[end_label, "DATETIME"] if "DATETIME" in df.columns else end_label,
                "adf_p_value": adf_series.loc[end_label],
                "kpss_p_value": kpss_series.loc[end_label] })

    results_df = pd.DataFrame(successful_windows)

    # удаляем колонки ADF и KPSS со значениями nan, оставляем только те где выполняются словия и по ADF и по KPSS
    df = df.loc[:, ~(df.columns.str.startswith('ADF ') | df.columns.str.startswith('KPSS ')) | df.notna().any()]
    return results_df, df

In [5]:
import warnings
from statsmodels.tools.sm_exceptions import (InterpolationWarning, ValueWarning)
warnings.filterwarnings('ignore', category=InterpolationWarning)
warnings.filterwarnings('ignore', category=ValueWarning)

pvalue_res, df_aug = stationarity_rolling_window(df=df, column_name="h-cleaned", min_pct=0.3, max_pct=0.7, step_pct=0.1, alpha_adf=0.05, alpha_kpss=0.05)
df_aug.head()

Анализ ряда длиной 1515


,TICKER,DATETIME,CLOSE,h-cleaned,ADF 454,KPSS 454,ADF 605,KPSS 605,ADF 756,KPSS 756,ADF 907,KPSS 907,ADF 1058,KPSS 1058
0,SiZ6SiH7 [FUTSPREAD],2026-10-01 07:00:00,1343.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,SiZ6SiH7 [FUTSPREAD],2026-10-01 07:15:00,1334.0,-9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,SiZ6SiH7 [FUTSPREAD],2026-10-01 07:30:00,1330.0,-4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,SiZ6SiH7 [FUTSPREAD],2026-10-01 07:45:00,1330.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,SiZ6SiH7 [FUTSPREAD],2026-10-01 08:00:00,1325.0,-5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
import re
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=df_aug['DATETIME'], y=df_aug['h-cleaned'], name=df_aug['TICKER'].iloc[1]), secondary_y=False)
# числовые суффиксы ADF
suffixes = sorted({match.group(1) for col in df_aug.columns if (match := re.fullmatch(r'ADF\s+(\d+)', col))}, key=int)  # подсказал ИИ

for suffix in suffixes:
    adf_col = f'ADF {suffix}'
    kpss_col = f'KPSS {suffix}'

    if kpss_col not in df_aug.columns:
        continue

    for col in [adf_col, kpss_col]:
        fig.add_trace(go.Scatter(x=df_aug['DATETIME'], y=df_aug[col], name=f'{col} rolling'), secondary_y=True)

fig.show()